In [5]:
from pymongo import MongoClient, GEOSPHERE

# 1. Connect to the remote MongoDB instance
client = MongoClient(
    host="mongo-csgy-6513-spring.db",
    port=27017,
    username="hm3424",
    password="bigdata",
    authSource="bigdata"
)
db = client.bigdata
restaurants = db.durham_restaurants
foreclosures = db.durham_foreclosures

# 2. Create 2dsphere indexes on the 'geometry' field in both collections
restaurants.create_index([("geometry", GEOSPHERE)])
foreclosures.create_index([("geometry", GEOSPHERE)])

print("✅ Connected to MongoDB and created 2dsphere indexes")


✅ Connected to MongoDB and created 2dsphere indexes


In [6]:
# Cell 2: Compute centroid for Food Service venues with at least 40 seats
# Aggregation pipeline:
#   1) match documents where Rpt_Area_Desc == "Food Service" and Seats >= 40
#   2) group to calculate average longitude and latitude from geometry.coordinates
pipeline = [
    {"$match": {
        "Rpt_Area_Desc": "Food Service",
        "Seats": {"$gte": 40}
    }},
    {"$group": {
        "_id": None,
        "avgLon": {"$avg": {"$arrayElemAt": ["$geometry.coordinates", 0]}},
        "avgLat": {"$avg": {"$arrayElemAt": ["$geometry.coordinates", 1]}}
    }}
]

centroid_doc = restaurants.aggregate(pipeline).next()
centroid_lon = centroid_doc["avgLon"]
centroid_lat = centroid_doc["avgLat"]

print(f"Centroid (lat, lon): ({centroid_lat:.6f}, {centroid_lon:.6f})")


Centroid (lat, lon): (35.970472, -78.915071)


In [7]:
# Cell 3: Count foreclosures within a 2-mile radius of the centroid
# 1 mile ≈ 1/3963.2 radians (Earth radius in miles)
earth_radius_miles = 3963.2
radius_radians = 2 / earth_radius_miles

# Geo-query using $geoWithin and $centerSphere
count_within_2_miles = foreclosures.count_documents({
    "geometry": {
        "$geoWithin": {
            "$centerSphere": [
                [centroid_lon, centroid_lat],
                radius_radians
            ]
        }
    }
})

print(f"Foreclosures within 2 miles: {count_within_2_miles}")



Foreclosures within 2 miles: 566
